# Document Scanner — Kaggle GPU Enhancement Network Loss Ablation (Resuming 20 -> 40 Epochs)

Continues training the 4-loss enhancement network ablation (`[REQ-45]`, ADR-006) on a Kaggle GPU (T4 / P100):

| Run | Loss Variant | Description | Config |
|---|---|---|---|
| exp-005 | L-A | MSE Loss | `configs/exp/exp-005_enh_mse.yaml` |
| exp-006 | L-B | L1 Loss | `configs/exp/exp-006_enh_l1.yaml` |
| exp-007 | L-C | L1 + MS-SSIM (alpha=0.84) | `configs/exp/exp-007_enh_l1msssim.yaml` |
| exp-008 | L-D | L1 + MS-SSIM + Sobel Edge (lambda=0.1) | `configs/exp/exp-008_enh_l1msssim_sobel.yaml` |

### Paired Shared-Stream Architecture (ADR-006)
`train_ablation.py` steps all 4 loss arms side-by-side on **one shared synthetic data stream**. This ensures:
1. **Maximum Speed**: Synthetic sample generation is done once per batch instead of 4 times.
2. **Scientific Rigour**: All 4 models receive identical batches in identical order from identical initial weights.
3. **Seamless Resumption**: By downloading/restoring your existing `DocEn_runs` folder (containing epoch 20 `last.pt` checkpoints), `--resume` continues all 4 arms from epoch 21 to 40.

### Step 0: Confirm GPU Accelerator Setup

In [ ]:
import os, sys, torch
print('PyTorch version:', torch.__version__)
print('vCPUs available:', os.cpu_count())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print('WARNING: CUDA GPU not detected. Ensure Accelerator is set to GPU P100 or T4 x2 in Kaggle settings.')

### Step 1: Clone Repository & Set Working Directory

In [ ]:
import os
repo_dir = '/kaggle/working/DocEn'
if not os.path.exists(repo_dir):
    !git clone https://github.com/HedieTahmouresi/DocEn.git {repo_dir}
%cd {repo_dir}
!git pull
!git log --oneline -1

### Step 2: Install Dependencies & Download / Extract Data

Paste your Google Drive File ID for `data.zip` in `DATA_ZIP_GDRIVE_ID` below to download directly via `gdown`, or attach `data.zip` as a Kaggle Dataset.

In [ ]:
!pip install -q -r requirements.txt gdown

import os, glob, shutil

# Optional: Set your Google Drive file ID for data.zip here
DATA_ZIP_GDRIVE_ID = ""

if not os.path.exists('data/clean_scans'):
    if DATA_ZIP_GDRIVE_ID:
        print('Downloading data.zip directly from Google Drive via gdown...')
        !gdown --id "{DATA_ZIP_GDRIVE_ID}" -O data.zip
        !unzip -q data.zip -d .
    else:
        # Search for data.zip in Kaggle input directories
        zip_candidates = glob.glob('/kaggle/input/**/data.zip', recursive=True) + ['/kaggle/working/data.zip']
        if zip_candidates:
            src_zip = zip_candidates[0]
            print('Extracting data from:', src_zip)
            !unzip -q "{src_zip}" -d .
        else:
            clean_scans_candidates = glob.glob('/kaggle/input/**/clean_scans', recursive=True)
            if clean_scans_candidates:
                data_root = os.path.dirname(clean_scans_candidates[0])
                print('Found unzipped data at:', data_root)
                !cp -r "{data_root}"/* data/
            else:
                print('WARNING: Provide DATA_ZIP_GDRIVE_ID above or attach data.zip as a Kaggle dataset.')

if not os.path.exists('data/frozen/val'):
    print('Frozen evaluation sets missing - generating them now...')
    !python -m src.data.freeze

!ls data && ls data/frozen

### Step 3: Restore Existing `DocEn_runs` Checkpoints from Google Drive Folder

Downloads the `DocEn_runs` folder directly from Google Drive into `runs/` using `gdown --folder`.

In [ ]:
import os, glob, shutil

# Google Drive folder URL for DocEn_runs
RUNS_GDRIVE_FOLDER = "https://drive.google.com/drive/folders/1cAxWn7GHuLCyWBLiq4-XCMhnnlyEiPtV"

os.makedirs('runs', exist_ok=True)

if RUNS_GDRIVE_FOLDER:
    print('Downloading DocEn_runs folder directly from Google Drive...')
    !gdown --folder "{RUNS_GDRIVE_FOLDER}" -O runs/ --remaining-ok
    # If gdown created a nested DocEn_runs subfolder inside runs, flatten it
    nested = glob.glob('runs/*/exp-005*') + glob.glob('runs/DocEn_runs/exp-005*')
    if nested:
        subfolder = os.path.dirname(nested[0])
        print('Flattening downloaded folder structure from:', subfolder)
        for exp in glob.glob(os.path.join(subfolder, 'exp-*')):
            target = os.path.join('runs', os.path.basename(exp))
            if os.path.exists(target):
                shutil.rmtree(target)
            shutil.move(exp, 'runs/')
    print('Restored runs:', sorted(os.listdir('runs')) if os.path.exists('runs') else [])
else:
    # Check for attached Kaggle Datasets containing runs or DocEn_runs
    runs_candidates = glob.glob('/kaggle/input/**/exp-005*', recursive=True)
    if runs_candidates:
        runs_parent = os.path.dirname(runs_candidates[0])
        print('Found existing checkpoint runs at:', runs_parent)
        for exp_dir in glob.glob(os.path.join(runs_parent, 'exp-*')):
            target_dir = os.path.join('runs', os.path.basename(exp_dir))
            if os.path.exists(target_dir):
                shutil.rmtree(target_dir)
            shutil.copytree(exp_dir, target_dir)
        print('Restored checkpoint runs:', sorted(os.listdir('runs')))
    else:
        print('No existing runs found in /kaggle/input/.')

if os.path.exists('runs'):
    !ls -la runs/

### Step 4: Run Sanity Unit Tests

In [ ]:
!python -m pytest tests/ -q -x

### Step 5: Resume 4-Loss Ablation Training (Epochs 21 -> 40)

Executes `--resume`, continuing `exp-005` .. `exp-008` seamlessly to epoch 40 on GPU.

In [ ]:
# Execute resumed training across all 4 arms
!python train_ablation.py --env colab_t4 --resume

### Step 6: Evaluate Restored Checkpoints & Generate Deliverables

Computes full evaluation metrics, generates loss curves (`p04_loss_curves.png`), loss comparison grids (`p04_loss_comparison.png`), and sample restored scans (`restored_samples/`).

In [ ]:
!python -m scripts.evaluate_ablation
!python -m scripts.save_restored_samples

### Step 7: Zip Output Results for Download

Packages all final checkpoints, evaluation metrics, and deliverable figures into `/kaggle/working/phase04_enhancement_results.zip`.

In [ ]:
!zip -r /kaggle/working/phase04_enhancement_results.zip runs/ outputs/figures/
print('Successfully created /kaggle/working/phase04_enhancement_results.zip')